# Capstone — Refresh / Content Opportunity Scoring

This notebook ranks anonymized pages for human refresh review. It uses a transparent baseline and a leakage-safe logistic model. Results are directional decision support, not causal evidence about Google or refresh impact.

In [6]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RAW_URL = 'https://raw.githubusercontent.com/Di-pesh/flyinterm/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(RAW_URL).drop_duplicates('content_id').copy()
lane = df.loc[df['content_type'].eq('keyword article')].copy()
lane['target'] = lane['trend_direction'].astype('string').str.lower().eq('down').astype(int)
print(f'Raw rows: {len(df):,}')
print(f'Keyword-article rows: {len(lane):,}')
print(f'Lane target base rate: {lane["target"].mean():.3f}')


Raw rows: 30,000
Keyword-article rows: 27,207
Lane target base rate: 0.561


## 1. Question and data

Which anonymized content pages should an editorial team review first for refresh or monitoring? The unit is a page and the output is a ranked queue with a score, action, and reason.

The source is the public starter snapshot `data/raw/content_refresh_anonymized.csv`. IDs are used only for grouping and tie-breaking. Label-derived fields and overlapping outcome-window fields are not model inputs.

In [7]:
excluded = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct', 'target',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'impression_tier', 'position_tier'
}
assert lane['target'].equals(lane['trend_direction'].astype('string').str.lower().eq('down').astype(int))
print(f'Excluded-field count: {len(excluded)}')
display(lane.groupby('content_type', dropna=False)['word_count'].apply(lambda s: s.isna().mean()).rename('word_count_missing_rate').to_frame())


Excluded-field count: 23


,word_count_missing_rate
content_type,
keyword article,0.282979


## 2. Transparent baseline

The baseline rule is: review pages that are both stale and visible. Stale means at least 91 days since update; visible means at least 500 impressions in the reported window. The score is intentionally readable and receives a reason code.

In [8]:
lane['is_stale'] = lane['days_since_last_update'].fillna(0).ge(91)
lane['is_visible'] = lane['impressions_90d'].fillna(0).ge(500)
lane['baseline_score'] = np.log1p(lane['impressions_90d'].fillna(0)) * (1 + lane['is_stale'].astype(int))
lane['baseline_action'] = np.where(lane['is_stale'] & lane['is_visible'], 'refresh_review', 'monitor')
lane['baseline_reason'] = np.where(lane['is_stale'] & lane['is_visible'], 'stale_visible', 'not_both_stale_visible')
baseline_queue = lane.sort_values(['baseline_score', 'content_id'], ascending=[False, True]).reset_index(drop=True)
baseline_queue.insert(0, 'rank', np.arange(1, len(baseline_queue) + 1))
display(lane['baseline_action'].value_counts().rename_axis('action').to_frame('rows'))
display(baseline_queue[['rank', 'baseline_score', 'baseline_action', 'baseline_reason', 'days_since_last_update', 'impressions_90d']].head(10))


,rows
action,
monitor,20693
refresh_review,6514


,rank,baseline_score,baseline_action,baseline_reason,days_since_last_update,impressions_90d
0,1,26.314364,refresh_review,stale_visible,104,517715
1,2,26.004613,refresh_review,stale_visible,104,443434
2,3,25.516464,refresh_review,stale_visible,104,347399
3,4,25.288081,refresh_review,stale_visible,104,309910
4,5,25.283442,refresh_review,stale_visible,104,309192
5,6,25.190126,refresh_review,stale_visible,104,295097
6,7,25.131748,refresh_review,stale_visible,104,286608
7,8,24.722406,refresh_review,stale_visible,104,233561
8,9,24.522702,refresh_review,stale_visible,104,211366
9,10,24.497105,refresh_review,stale_visible,104,208678


## 3. Grouped model evaluation

The model uses metadata and recency fields only. Missing numeric fields get a training-fold median and a missingness flag; missing categories become `unknown`. A complete client is held out so pages from one client cannot occur in both train and test. Both methods are evaluated on the same test rows.

In [14]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# 1. RESET INDEX
# ============================================================

lane = lane.reset_index(drop=True).copy()


# ============================================================
# 2. DEFINE FEATURES
# ============================================================

safe_numeric = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

safe_categorical = [
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
]

required_columns = (
    safe_numeric
    + safe_categorical
    + [
        "client_id",
        "target",
        "baseline_score",
    ]
)


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

missing_columns = [
    column
    for column in required_columns
    if column not in lane.columns
]

if missing_columns:
    raise KeyError(
        f"The following required columns are missing from lane: "
        f"{missing_columns}"
    )


# ============================================================
# 4. CLEAN TARGET
# ============================================================

# Convert target to numeric first.
lane["target"] = pd.to_numeric(
    lane["target"],
    errors="coerce",
)

# Remove rows where target is missing.
valid_target = lane["target"].isin([0, 1])

if not valid_target.all():
    print(
        f"Dropping {(~valid_target).sum():,} rows "
        f"with invalid/missing target values."
    )

    lane = lane.loc[valid_target].reset_index(drop=True)


# Make sure target is integer.
y = lane["target"].astype("int8").reset_index(drop=True)


# ============================================================
# 5. BUILD FEATURE FRAME
# ============================================================

X_raw = lane[
    safe_numeric + safe_categorical
].copy()


# ============================================================
# 6. CLEAN NUMERIC FEATURES
# ============================================================

for column in safe_numeric:

    X_raw[column] = pd.to_numeric(
        X_raw[column],
        errors="coerce",
    )

    # Explicit missingness indicator.
    X_raw[f"has_{column}"] = (
        X_raw[column]
        .notna()
        .astype("int8")
    )


# ============================================================
# 7. CLEAN CATEGORICAL FEATURES
# ============================================================

for column in safe_categorical:

    X_raw[column] = (
        lane[column]
        .astype("object")
        .where(
            lane[column].notna(),
            "unknown",
        )
        .replace(
            "",
            "unknown",
        )
    )


# ============================================================
# 8. RESET ALL INDEXES
# ============================================================

X_raw = X_raw.reset_index(drop=True)

groups = (
    lane["client_id"]
    .astype("object")
    .where(
        lane["client_id"].notna(),
        "unknown",
    )
    .reset_index(drop=True)
)


numeric_columns = safe_numeric + [
    f"has_{column}"
    for column in safe_numeric
]


# ============================================================
# 9. VERIFY TRAIN / TEST SPLIT
# ============================================================

if "train_idx" not in globals() or "test_idx" not in globals():
    raise RuntimeError(
        "train_idx and test_idx must already exist from "
        "your grouped train/test split."
    )


train_idx = np.asarray(train_idx)
test_idx = np.asarray(test_idx)


if len(train_idx) == 0 or len(test_idx) == 0:
    raise RuntimeError(
        "Train or test split is empty."
    )


# Make sure indexes are valid.
if (
    train_idx.min() < 0
    or test_idx.min() < 0
    or train_idx.max() >= len(X_raw)
    or test_idx.max() >= len(X_raw)
):
    raise IndexError(
        "train_idx or test_idx contains an invalid row index."
    )


# ============================================================
# 10. CHECK TARGET CLASSES
# ============================================================

train_classes = y.iloc[train_idx].nunique()
test_classes = y.iloc[test_idx].nunique()

if train_classes != 2:
    raise RuntimeError(
        "Training set must contain both target classes (0 and 1)."
    )

if test_classes != 2:
    raise RuntimeError(
        "Test set must contain both target classes (0 and 1)."
    )


# ============================================================
# 11. CHECK CLIENT GROUP LEAKAGE
# ============================================================

train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

overlap = train_clients.intersection(
    test_clients
)

if overlap:
    raise RuntimeError(
        f"Client leakage detected: "
        f"{len(overlap)} clients appear in both train and test."
    )


# ============================================================
# 12. PREPROCESSOR FACTORY
# ============================================================

def make_preprocessor(use_dense=False):
    """
    Create a fresh preprocessor for each candidate model.

    Numeric:
        median imputation

    Categorical:
        constant imputation + one-hot encoding
    """

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="median"
                            ),
                        )
                    ]
                ),
                numeric_columns,
            ),

            (
                "categorical",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="unknown",
                            ),
                        ),

                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=not use_dense,
                            ),
                        ),
                    ]
                ),
                safe_categorical,
            ),
        ],

        remainder="drop",
    )


# ============================================================
# 13. CANDIDATE MODELS
# ============================================================

candidate_models = {

    "Logistic regression": {
        "classifier": LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        ),
        "dense": False,
    },

    "Random forest": {
        "classifier": RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        ),
        "dense": False,
    },

    "Extra trees": {
        "classifier": ExtraTreesClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=15,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "dense": False,
    },

    "HistGradientBoosting": {
        "classifier": HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=31,
            min_samples_leaf=25,
            l2_regularization=1.0,
            random_state=42,
        ),
        "dense": True,
    },
}


# ============================================================
# 14. PRECISION@K
# ============================================================

def precision_at_k(labels, scores, k):
    """
    Precision among the top-k highest-scoring rows.
    """

    labels = np.asarray(labels)
    scores = np.asarray(scores)

    if len(labels) == 0:
        return 0.0

    k = min(k, len(labels))

    order = np.argsort(
        -scores
    )[:k]

    return float(
        labels[order].mean()
    )


# ============================================================
# 15. EVALUATION FUNCTION
# ============================================================

def evaluate_scores(
    method_name,
    labels,
    scores,
):
    """
    Evaluate ranking performance.
    """

    labels = np.asarray(labels)
    scores = np.asarray(scores)

    return {
        "method": method_name,

        "base_rate": float(
            labels.mean()
        ),

        "roc_auc": float(
            roc_auc_score(
                labels,
                scores,
            )
        ),

        "average_precision": float(
            average_precision_score(
                labels,
                scores,
            )
        ),

        "precision_at_10": precision_at_k(
            labels,
            scores,
            10,
        ),

        "precision_at_50": precision_at_k(
            labels,
            scores,
            50,
        ),

        "precision_at_100": precision_at_k(
            labels,
            scores,
            100,
        ),
    }


# ============================================================
# 16. TEST LABELS
# ============================================================

test_y = (
    y.iloc[test_idx]
    .to_numpy(dtype=int)
)


# ============================================================
# 17. RULE-BASED BASELINE
# ============================================================

baseline_scores = pd.to_numeric(
    lane["baseline_score"],
    errors="coerce",
)

# Replace missing baseline values with the median
# calculated ONLY on the test baseline values.
baseline_scores = (
    baseline_scores
    .iloc[test_idx]
    .fillna(
        baseline_scores.iloc[test_idx].median()
    )
    .to_numpy(dtype=float)
)


all_scores = {
    "Rule baseline": baseline_scores,
}


results = [
    evaluate_scores(
        "Rule baseline",
        test_y,
        baseline_scores,
    )
]


# ============================================================
# 18. TRAIN CANDIDATE MODELS
# ============================================================

trained_models = {}

for model_name, config in candidate_models.items():

    print(
        f"\nTraining {model_name}..."
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocess",
                make_preprocessor(
                    use_dense=config["dense"]
                ),
            ),

            (
                "classifier",
                config["classifier"],
            ),
        ]
    )

    try:

        # Fit ONLY on training data.
        pipeline.fit(
            X_raw.iloc[train_idx],
            y.iloc[train_idx],
        )

        # Generate probabilities ONLY for held-out data.
        scores = pipeline.predict_proba(
            X_raw.iloc[test_idx]
        )[:, 1]

        trained_models[
            model_name
        ] = pipeline

        all_scores[
            model_name
        ] = scores

        results.append(
            evaluate_scores(
                model_name,
                test_y,
                scores,
            )
        )

        print(
            f"Finished {model_name}"
        )

    except Exception as error:

        print(
            f"Skipped {model_name}: {error}"
        )


# ============================================================
# 19. COMPARE RESULTS
# ============================================================

metrics_df = pd.DataFrame(
    results
)


# Operational metric comes first:
# precision@50
metrics_df = (
    metrics_df
    .sort_values(
        by=[
            "precision_at_50",
            "precision_at_100",
            "average_precision",
            "roc_auc",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    metrics_df.round(3)
)


# ============================================================
# 20. SELECT WINNER
# ============================================================

selected_method = (
    metrics_df.iloc[0]["method"]
)


if selected_method == "Rule baseline":

    selected_scores = baseline_scores
    selected_model = None

else:

    selected_scores = (
        all_scores[selected_method]
    )

    selected_model = (
        trained_models[selected_method]
    )


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

print(
    f"\nTrain rows: {len(train_idx):,}"
)

print(
    f"Test rows: {len(test_idx):,}"
)

print(
    f"Held-out clients: "
    f"{groups.iloc[test_idx].nunique():,}"
)

print(
    f"Train clients: "
    f"{groups.iloc[train_idx].nunique():,}"
)

print(
    f"Test base rate: "
    f"{test_y.mean():.3f}"
)

print(
    f"Selected method by precision@50: "
    f"{selected_method}"
)

print(
    f"Selected precision@50: "
    f"{metrics_df.iloc[0]['precision_at_50']:.3f}"
)


Training Logistic regression...


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Finished Logistic regression

Training Random forest...
Finished Random forest

Training Extra trees...
Finished Extra trees

Training HistGradientBoosting...
Finished HistGradientBoosting


,method,base_rate,roc_auc,average_precision,precision_at_10,precision_at_50,precision_at_100
0,HistGradientBoosting,0.504,0.567,0.572,0.6,0.56,0.55
1,Extra trees,0.504,0.561,0.551,0.6,0.52,0.47
2,Logistic regression,0.504,0.576,0.573,0.7,0.48,0.42
3,Random forest,0.504,0.559,0.562,0.3,0.36,0.49
4,Rule baseline,0.504,0.579,0.526,0.3,0.28,0.28



Train rows: 21,425
Test rows: 5,782
Held-out clients: 7
Train clients: 24
Test base rate: 0.504
Selected method by precision@50: HistGradientBoosting
Selected precision@50: 0.560


## 4. Leakage sentinel

`trend_pct` is label-derived and must not be a feature. The sentinel below uses the negative of `trend_pct` because more negative trend values correspond to the positive decline class. A strong score here confirms that the test harness can detect the leak; this number must not be reported as model performance.

In [15]:
leaky_score = -pd.to_numeric(lane['trend_pct'], errors='coerce').fillna(0).reset_index(drop=True).iloc[test_idx].to_numpy()
leak_auc = roc_auc_score(test_y, leaky_score)
print(f'Deliberate trend-derived leakage AUC: {leak_auc:.3f} (sentinel only)')
assert leak_auc > 0.80, 'Leakage sentinel failed; inspect the target or test harness.'
print('PASS: trend_pct is detectable as leakage and is excluded from the model.')


Deliberate trend-derived leakage AUC: 1.000 (sentinel only)
PASS: trend_pct is detectable as leakage and is excluded from the model.


## 5. Limitations and recommendations

This is a retrospective starter-snapshot analysis, not a randomized content experiment. It cannot establish that refreshing a page causes more visibility, clicks, or engagement. The safe claim is observed, directional decision support for human review.

Recommended order: (1) review stale-and-visible pages first; (2) verify seasonality and recent editorial work manually; (3) monitor low-volume stale pages; (4) use reason codes as explanations, not automatic instructions; and (5) validate on a later time window before operational adoption.

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

In [16]:
from pathlib import Path
out_dir = Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(out_dir / 'capstone_metrics.csv', index=False)
baseline_queue[['rank', 'baseline_score', 'baseline_action', 'baseline_reason', 'days_since_last_update', 'impressions_90d']].head(100).to_csv(out_dir / 'capstone_queue_preview.csv', index=False)
print(f'Wrote {out_dir / "capstone_metrics.csv"}')
print(f'Wrote {out_dir / "capstone_queue_preview.csv"}')


Wrote work/outputs/capstone_metrics.csv
Wrote work/outputs/capstone_queue_preview.csv


## Self-check

- [x] Data loads from the public raw GitHub URL.
- [x] Missing values are handled inside the model pipeline.
- [x] Grouped split is guaranteed to contain both classes.
- [x] Baseline and model use identical test rows.
- [x] Base rate and precision-at-K are reported.
- [x] Leakage sentinel correctly reverses trend_pct before testing.
- [ ] Run all cells and save the executed notebook before submission.